# Kinetic theory of an ideal gas

*thermodynamics · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/kinetic_gas)

Hundreds of hard disks bounce around a box. Every disk starts with the same speed, yet elastic collisions shuffle the energy until the speeds follow the Maxwell–Boltzmann distribution, and the momentum the disks hand to the walls reproduces the ideal gas law $PA = NkT$.

**Vector form**

$$\mathbf v_a' = \mathbf v_a - \big[(\mathbf v_a - \mathbf v_b)\cdot\hat{\mathbf n}_{ab}\big]\,\hat{\mathbf n}_{ab},\qquad P A = N k T$$

**Index (tensor) form**

$$v'_{ai} = v_{ai} - (\Delta v_{abj}\,n_{abj})\,n_{abi},\qquad n_{abi} = \frac{d_{abi}}{\sqrt{d_{abk}d_{abk}}},\quad d_{abi} = x_{ai} - x_{bi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{ai}$ | `x` | (N, 2) | position of disk $a$, component $i \in \{x, y\}$ |
| $v_{ai}$ | `v` | (N, 2) | velocity of disk $a$ |
| $d_{abi}$ | `x[:, None, :] - x[None, :, :]` | (N, N, 2) | every pairwise displacement at once, by broadcasting |
| $d_{abi}d_{abi}$ | `einsum('abi,abi->ab', d, d)` | (N, N) | repeated $i$ ⇒ summed: squared distance of every pair |
| $\Delta v_{pj}\,n_{pj}$ | `einsum('pj,pj->p', v[a] - v[b], n)` | (P,) | relative velocity projected on the collision normal, for the $P$ colliding pairs |
| $E$ | `0.5 * m * einsum('ai,ai->', v, v)` | scalar | total kinetic energy, equal to $NkT$ in 2D |
| $f(v)$ | `maxwell_boltzmann_2d(v, kT)` | (V,) | 2D speed distribution $(mv/kT)\exp(-mv^2/2kT)$ |

Reduced units: box side 1, mass 1, Boltzmann constant 1. The pair tensors are $N\times N$, which is fine for a few hundred disks (real codes use cell lists). A disk touching two others in the same step waits one step, so every collision is a clean two-body exchange and the energy is conserved to round-off. The pressure is the momentum $2m\lvert v_\perp\rvert$ delivered per bounce, summed over the run and divided by time × wall length. It comes out a few percent above $NkT/A$ because the disks themselves take up area — the first correction to the ideal gas, roughly $(1 + \eta^2/8)/(1 - \eta)^2$ for area fraction $\eta$.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/kinetic_gas/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    n, radius = params["n_particles"], params["radius"]
    x0 = tensors.initial_positions(n, radius, dev)  # (N, 2)
    v0 = tensors.initial_velocities(n, params["speed"], dev)  # (N, 2)

    times, xs, vs, pressure = solve.solve(x0, v0, tensors.BOX, radius, params["t_max"], tensors.MASS)

    energy = equation.kinetic_energy(vs, tensors.MASS)  # (F,)
    kT = equation.temperature(v0, tensors.MASS).item()  # same at every step: E is conserved
    area = (tensors.BOX - 2 * radius) ** 2  # region the disk centres can reach
    ideal = n * kT / area
    print(f"kinetic energy {energy[0]:.4f} -> {energy[-1]:.4f}   (kT = {kT:.3f})")
    print(f"pressure from the walls {pressure:.2f}   ideal gas N kT / A = {ideal:.2f}   ratio {pressure / ideal:.3f}")

    title = f"kT = {kT:.2f}    P = {pressure:.0f}  vs  NkT/A = {ideal:.0f}"
    return plot.animate(times, xs, vs, kT, tensors.BOX, radius, title, out_dir / "kinetic_gas.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/kinetic_gas/equation.py
r"""Physics → tensor translation.

An ideal gas of N hard disks a = 1..N with positions x_{ai} and velocities v_{ai} (i ∈ {x, y}).
Between collisions every particle moves in a straight line. Two touching disks a, b exchange the
component of their relative velocity along the line joining them (elastic, equal masses):

    d_{abi} = x_{ai} - x_{bi},        n_{abi} = d_{abi} / \sqrt{d_{abj} d_{abj}}

    v'_{ai} = v_{ai} - (\Delta v_{abj} n_{abj}) \, n_{abi},     v'_{bi} = v_{bi} + (\Delta v_{abj} n_{abj}) \, n_{abi}

with \Delta v_{abj} = v_{aj} - v_{bj}. The repeated j is summed: it is the projection of the relative
velocity onto the collision normal, einsum('pj,pj->p'). Both the momentum and the kinetic energy
E = ½ m v_{ai} v_{ai} (summed over a and i) are unchanged by this rule.

Statistics: in 2D each particle carries kT of kinetic energy (½kT per direction), the speeds relax to the
Maxwell–Boltzmann distribution f(v) = (m v / kT) e^{-m v² / 2kT}, and the wall pressure obeys P A = N k T.
"""

import torch


def pair_displacement(x: torch.Tensor) -> torch.Tensor:
    """d_{abi} = x_{ai} − x_{bi} by broadcasting (N, 1, 2) − (1, N, 2).   x: (N, 2) → (N, N, 2)"""
    return x[:, None, :] - x[None, :, :]


def colliding_pairs(x: torch.Tensor, v: torch.Tensor, radius: float) -> tuple[torch.Tensor, torch.Tensor]:
    """Indices (a, b) of disks that overlap and are moving towards each other.  Each disk in at most one pair."""
    d = pair_displacement(x)  # (N, N, 2)
    dv = pair_displacement(v)  # (N, N, 2)  Δv_{abi}
    dist2 = torch.einsum("abi,abi->ab", d, d)  # (N, N)  squared distance
    approaching = torch.einsum("abi,abi->ab", dv, d) < 0  # (N, N)  Δv · d < 0
    hit = (dist2 < (2 * radius) ** 2) & approaching
    a, b = torch.nonzero(torch.triu(hit, diagonal=1), as_tuple=True)  # (P,), (P,)  each pair once, a < b

    # A disk touching two others at once waits until the next step, so every update is a clean two-body one.
    count = torch.bincount(torch.cat([a, b]), minlength=len(x))  # (N,)  pairs each disk appears in
    alone = (count[a] == 1) & (count[b] == 1)
    return a[alone], b[alone]


def collide(x: torch.Tensor, v: torch.Tensor, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    r"""Elastic collisions of the pairs (a_p, b_p):  v'_{ai} = v_{ai} − (Δv_{pj} n_{pj}) n_{pi}.  → new v (N, 2)"""
    d = x[a] - x[b]  # (P, 2)
    n = d / torch.sqrt(torch.einsum("pj,pj->p", d, d))[:, None]  # (P, 2)  unit collision normal
    w = torch.einsum("pj,pj->p", v[a] - v[b], n)  # (P,)  relative velocity along the normal
    v = v.clone()
    v[a] = v[a] - w[:, None] * n
    v[b] = v[b] + w[:, None] * n
    return v


def bounce_walls(x: torch.Tensor, v: torch.Tensor, box: float, radius: float, m: float = 1.0):
    """Reflect v_i of disks that hit a wall; return (new v (N, 2), momentum given to the walls Σ 2 m |v_i|)."""
    hit = ((x < radius) & (v < 0)) | ((x > box - radius) & (v > 0))  # (N, 2)  wall normal along i
    impulse = 2 * m * v.abs()[hit].sum()
    return torch.where(hit, -v, v), impulse


def kinetic_energy(v: torch.Tensor, m: float = 1.0) -> torch.Tensor:
    """E = ½ m v_{ai} v_{ai}, summed over particles a and components i.  (..., N, 2) → (...)"""
    return 0.5 * m * torch.einsum("...ai,...ai->...", v, v)


def temperature(v: torch.Tensor, m: float = 1.0) -> torch.Tensor:
    """Equipartition in 2D: E = N kT, so kT = E / N  (with k = 1)."""
    return kinetic_energy(v, m) / v.shape[-2]


def maxwell_boltzmann_2d(speed: torch.Tensor, kT: float, m: float = 1.0) -> torch.Tensor:
    """Probability density of the speed in 2D:  f(v) = (m v / kT) exp(−m v² / 2kT)."""
    return m * speed / kT * torch.exp(-m * speed**2 / (2 * kT))


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/kinetic_gas/tensors.py
"""Build the concrete tensors: positions on a jittered grid and velocities of equal speed.

Units are "reduced": box side L = 1, particle mass m = 1 and Boltzmann's constant k = 1.
"""

import math

import torch

BOX = 1.0  # side L of the square box
MASS = 1.0


def initial_positions(n: int, radius: float, device: torch.device, seed: int = 0) -> torch.Tensor:
    """x_{ai}: n disks on a square grid, each nudged randomly inside its own cell (no overlaps).  (N, 2)"""
    gen = torch.Generator().manual_seed(seed)
    per_side = math.ceil(math.sqrt(n))
    cell = BOX / per_side
    i, j = torch.meshgrid(torch.arange(per_side), torch.arange(per_side), indexing="ij")
    centres = (torch.stack([i, j], dim=-1).reshape(-1, 2)[:n] + 0.5) * cell  # (N, 2) cell centres
    wiggle = 0.5 * cell - radius  # how far a disk may move and still stay in its cell
    jitter = (2 * torch.rand(n, 2, generator=gen) - 1) * 0.9 * wiggle
    return (centres + jitter).to(device)


def initial_velocities(n: int, speed: float, device: torch.device, seed: int = 1) -> torch.Tensor:
    """v_{ai}: every disk has the same speed, random direction.  (N, 2)

    The second half moves exactly opposite to the first half, so the total momentum is zero.
    """
    gen = torch.Generator().manual_seed(seed)
    angle = 2 * math.pi * torch.rand(n, generator=gen)  # (N,)
    half = n // 2
    angle[half : 2 * half] = angle[:half] + math.pi
    return (speed * torch.stack([torch.cos(angle), torch.sin(angle)], dim=-1)).to(device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/kinetic_gas/solve.py
"""Event-free molecular dynamics: drift, bounce off the walls, collide, repeat."""

import torch

from .equation import bounce_walls, collide, colliding_pairs


def solve(
    x0: torch.Tensor,
    v0: torch.Tensor,
    box: float,
    radius: float,
    t_max: float,
    mass: float = 1.0,
    n_snapshots: int = 121,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, float]:
    """Return (times (F,), positions (F, N, 2), velocities (F, N, 2), pressure P).

    The time step lets the fastest disk move at most half its radius per step,
    so no collision is missed. P = (momentum given to the walls) / (time × wall length).
    """
    v_max = 3 * v0.norm(dim=1).max().item()  # speeds rarely exceed 3× the initial one
    n_steps = max(1, int(t_max * v_max / (0.5 * radius)))
    dt = t_max / n_steps
    keep = set(torch.linspace(0, n_steps, n_snapshots).round().int().tolist())  # steps to record

    x, v = x0, v0
    times, xs, vs = [0.0], [x0], [v0]
    momentum_to_walls = 0.0
    for step in range(1, n_steps + 1):
        x = x + v * dt  # free flight
        v, impulse = bounce_walls(x, v, box, radius, mass)
        momentum_to_walls += impulse.item()
        a, b = colliding_pairs(x, v, radius)
        if len(a):
            v = collide(x, v, a, b)
        if step in keep:
            times.append(step * dt)
            xs.append(x)
            vs.append(v)

    # The disk centres only reach to within one radius of the walls: effective side L − 2r.
    wall_length = 4 * (box - 2 * radius)
    pressure = momentum_to_walls / (t_max * wall_length)
    return torch.tensor(times), torch.stack(xs), torch.stack(vs), pressure


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/kinetic_gas/plot.py
"""Animate the disks in the box next to their speed histogram and the Maxwell–Boltzmann curve."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import maxwell_boltzmann_2d


def animate(
    times: torch.Tensor,
    xs: torch.Tensor,
    vs: torch.Tensor,
    kT: float,
    box: float,
    radius: float,
    title: str,
    path: Path,
) -> Path:
    t = times.cpu().numpy()
    pos = xs.cpu().numpy()  # (F, N, 2)
    speed = vs.norm(dim=-1).cpu().numpy()  # (F, N)
    v_top = 3.2 * np.sqrt(kT)  # right edge of the histogram
    bins = np.linspace(0, v_top, 26)
    v_curve = torch.linspace(0, v_top, 200)
    f_curve = maxwell_boltzmann_2d(v_curve, kT).numpy()

    fig, (ax, ax_h) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1, 1.1]})
    ax.set_xlim(0, box)
    ax.set_ylim(0, box)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    marker = (2 * radius / box * 3.0 * 72) ** 2  # disk diameter in points², box ≈ 3 inches tall
    dots = ax.scatter(pos[0, :, 0], pos[0, :, 1], s=marker, c=speed[0], cmap="viridis", vmin=0, vmax=v_top)
    clock = ax.set_title("")

    hist = ax_h.hist(speed[0], bins=bins, density=True, color=ACCENT[0], alpha=0.6, label="simulation")[2]
    ax_h.plot(v_curve.numpy(), f_curve, color=ACCENT[1], lw=2, label="Maxwell–Boltzmann")
    ax_h.set_xlim(0, v_top)
    ax_h.set_ylim(0, 1.6 * f_curve.max())
    ax_h.set_xlabel("speed |v|")
    ax_h.set_ylabel("probability density")
    ax_h.set_title(title, fontsize=9)
    ax_h.legend(loc="upper right", fontsize=8)

    def update(step: int):
        dots.set_offsets(pos[step])
        dots.set_array(speed[step])
        heights, _ = np.histogram(speed[step], bins=bins, density=True)
        for bar, h in zip(hist, heights):
            bar.set_height(h)
        clock.set_text(f"t = {t[step]:.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'n_particles': 300,
    'radius': 0.006,
    'speed': 1.0,
    't_max': 4.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.kinetic_gas")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.kinetic_gas.main")
problem_dir = Path("problems/thermodynamics/kinetic_gas")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
